In [97]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

from typing import TypedDict


In [98]:
llm = ChatOllama(
    model= "llama3.2:1b"
)

In [99]:
from typing import Annotated
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage
from langgraph.graph.message import add_messages


class State(TypedDict):
    question: str
    answer: str
    route: str
    messages: Annotated[list[AnyMessage], add_messages]

In [100]:
@tool
def maths(a: float , b: float) -> float:
    """
        You are a maths addition.
        You always perform addition of 2 numbers.
        a: first number
        b: first number

        result: addition of above numbers
    """

    return a+b

llm_with_tools =  llm.bind_tools(tools=[maths])




In [101]:
def tool_agent(state: State):
    question = state['question']
    res = llm_with_tools.invoke(([
        SystemMessage(content="You are a maths assistant. Always use the maths tool for addition."),
        HumanMessage(content=question)
    ]))

    return {"messages": [res]}

def normalFlow(State: State):
    question = State['question']

    res = llm.invoke(question)
    return {"answer" : res.content}
    

In [102]:
from langgraph.prebuilt import ToolNode


def router(state: State):
    question = state["question"]

    router_prompt = f"""
You are a query router. Your job is to decide which workflow should handle the user's question.

Available routes:

TOOL:
- Use this route whenever the user asks for mathematical addition.
- Examples: 5 + 5, add 10 and 20, what is the sum of 15 and 25.

Normal:
- Use this route for all other questions.
- Examples: general knowledge, company policies, explanations, etc.

Rules:
- Return ONLY one word: TOOL or Normal.
- Do not solve the user's question.
- Do not provide explanations.

User question: {question}
"""

    res = llm.invoke(router_prompt)

    decision = res.content.strip().lower()

    if "tool" in decision:
        return {"route": "tool"}
    return {"route": "normal"}

def normal(state: State):
    question = state["question"]

    res = llm.invoke(question)

    return {"answer" : res.content}

tool_node = ToolNode([maths])

In [107]:
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import tools_condition

graph = StateGraph(State)

graph.add_node("router", router)
graph.add_node("tool_agent", tool_agent)
graph.add_node("normal", normal)
graph.add_node("tools", tool_node)

graph.add_edge(START, "router")
graph.add_conditional_edges("router", 
  lambda state: state["route"]                          
, 
{
    "tool" : "tool_agent",
    "normal" : "normal"
})

graph.add_edge("normal", END)

graph.add_conditional_edges(
    "tool_agent",
    tools_condition
)

graph.add_edge("tools", "tool_agent")


workflow = graph.compile()

res = workflow.invoke({
    "question": "what is 5 plus 5",
    "messages": []
})
res

{'question': 'what is 5 plus 5',
 'route': 'tool',
 'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-11-11T16:16:45.755430433Z', 'done': True, 'done_reason': 'stop', 'total_duration': 86278729, 'load_duration': 809149, 'prompt_eval_count': 204, 'prompt_eval_duration': 10385000, 'eval_count': 21, 'eval_duration': 73119000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a1cb65-c064-73e0-8421-13c0a2203a7b-0', tool_calls=[{'name': 'maths', 'args': {'function': 'maths', 'parameters': {'a': 5, 'b': 5}, 'type': 'function'}, 'id': '399a5b1b-b3ab-40d7-b1ec-a64bd5e79999', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'output_tokens': 21, 'total_tokens': 225}),
  ToolMessage(content="Error invoking tool 'maths' with kwargs {'function': 'maths', 'parameters': {'a': 5, 'b': 5}, 'type': 'function'} with error:\n a: Field required\nb: Field required\n 

In [105]:
router({"question": "what is 5 plus 5"})

{'route': 'tool'}